# Nutrition CallBot — Colab Runtime
**Stack:** Gemini 2.5 Flash · E5-large-instruct · Qdrant local

In [ ]:
# CELL 1: Clone repo
!git clone https://github.com/enigmaticcat/legal-voice-callbot.git
%cd /content/legal-voice-callbot
!git pull


In [ ]:
# CELL 2: Cài đặt dependencies (không cần vLLM)
!pip install -q \n    google-genai>=1.0.0 \n    sentence-transformers>=3.0.0 \n    qdrant-client>=1.7.0 \n    fastapi>=0.115.0 \n    "uvicorn[standard]>=0.30.0" \n    pydantic>=2.9.0 \n    python-dotenv>=1.0.0


In [ ]:
# CELL 3: Config — đọc từ Colab Secrets (Wrench icon → Secrets)
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# Qdrant chạy local trên Colab
os.environ["QDRANT_URL"]     = "http://localhost:6333"
os.environ["QDRANT_API_KEY"] = ""

print("GEMINI_API_KEY:", os.environ["GEMINI_API_KEY"][:8] + "...")
print("QDRANT_URL    :", os.environ["QDRANT_URL"])


In [ ]:
# CELL 4: Khởi động Qdrant local
import subprocess, time

!curl -sL https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-musl.tar.gz \n    | tar -xz

qdrant_proc = subprocess.Popen(
    ["/content/qdrant"],
    stdout=open("qdrant.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

import requests
r = requests.get("http://localhost:6333/healthz")
print(f"Qdrant: {r.status_code} (PID={qdrant_proc.pid})")


In [ ]:
# CELL 5: Mount Google Drive + restore Qdrant snapshot
from google.colab import drive
drive.mount("/content/drive")

SNAPSHOT_PATH = "/content/drive/MyDrive/Nutrition data/nutrition_articles-2744933042503761-2026-03-30-08-11-07.snapshot"
COLLECTION    = "nutrition_articles"

import requests

# Xóa collection cũ nếu tồn tại
requests.delete(f"http://localhost:6333/collections/{COLLECTION}")

# Upload snapshot
print("Uploading snapshot (có thể mất 1-2 phút)...")
r = requests.post(
    f"http://localhost:6333/collections/{COLLECTION}/snapshots/upload?priority=snapshot",
    files={"snapshot": open(SNAPSHOT_PATH, "rb")},
    timeout=300,
)
print(f"Status: {r.status_code}")

# Kiểm tra
info = requests.get(f"http://localhost:6333/collections/{COLLECTION}").json()
print(f"Collection: {COLLECTION} | vectors: {info['result']['vectors_count']}")


In [ ]:
# CELL 6: Khởi động Brain server (Gemini)
import subprocess, time, requests, sys

PROJECT = "/content/legal-voice-callbot/nutrition-callbot"

# Kill server cũ nếu có
import os; os.system("pkill -f brain.server 2>/dev/null")
time.sleep(2)

brain_proc = subprocess.Popen(
    [sys.executable, "-m", "brain.server"],
    cwd=PROJECT,
    stdout=open("brain.log", "w"),
    stderr=subprocess.STDOUT,
)

# Chờ server sẵn sàng
for i in range(60):
    try:
        if requests.get("http://localhost:50052/health", timeout=2).status_code == 200:
            print(f"Brain server OK (PID={brain_proc.pid})")
            break
    except: pass
    time.sleep(2)
    if i == 59: print("TIMEOUT — xem brain.log")

# In 10 dòng log cuối
!tail -10 /content/brain.log


In [ ]:
# CELL 7: Test single query
import requests, json, time

QUERY = "Thưa bác sĩ, tôi muốn hỏi có nên uống Omega 3-6-9 mỗi ngày hay không?"

t0 = time.time()
r = requests.post(
    "http://localhost:50052/think/stream",
    json={"query": QUERY, "session_id": "test"},
    stream=True,
)

full_text = ""
ttft = None
for line in r.iter_lines():
    if not line: continue
    data = json.loads(line.decode())
    if data.get("text") and not data.get("is_final"):
        if ttft is None:
            ttft = time.time() - t0
        full_text += data["text"]
    if data.get("is_final"):
        timing = data.get("timing", {})

print(f"Query : {QUERY}")
print(f"Answer: {full_text}")
print(f"TTFT  : {ttft:.2f}s | Total: {time.time()-t0:.2f}s")


In [ ]:
# CELL 8: Batch evaluation trên thucuc_qa (255 câu)
import requests, json, time

QA_PATH = "/content/legal-voice-callbot/evaluation/thucuc_qa.jsonl"

results = []
with open(QA_PATH, encoding="utf-8") as f:
    records = [json.loads(l) for l in f if l.strip()]

print(f"Bắt đầu evaluate {len(records)} câu...")

for i, rec in enumerate(records):
    t0 = time.time()
    try:
        r = requests.post(
            "http://localhost:50052/think/stream",
            json={"query": rec["question"], "session_id": f"eval_{i}"},
            stream=True, timeout=60,
        )
        answer = ""
        contexts = []
        timing = {}
        ttft_s = None
        for line in r.iter_lines():
            if not line: continue
            d = json.loads(line.decode())
            if d.get("text") and not d.get("is_final"):
                if ttft_s is None:
                    ttft_s = time.time() - t0
                answer += d["text"]
            if d.get("is_final"):
                contexts = d.get("contexts", [])
                timing   = d.get("timing", {})
        results.append({
            "question"    : rec["question"],
            "answer"      : answer,
            "ground_truth": rec["answer"],
            "contexts"    : contexts,
            "timing"      : timing,
            "ttft_s"      : round(ttft_s or 0, 3),
            "latency_s"   : round(time.time() - t0, 2),
        })
    except Exception as e:
        results.append({"question": rec["question"], "error": str(e)})

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(records)} done")

# Lưu kết quả
OUT = "/content/drive/MyDrive/Nutrition data/eval_results.jsonl"
with open(OUT, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

errors  = sum(1 for r in results if "error" in r)
avg_lat = sum(r.get("latency_s", 0) for r in results if "error" not in r) / max(1, len(results) - errors)
print(f"Done: {len(results)} | Errors: {errors} | Avg latency: {avg_lat:.1f}s")
print(f"Saved: {OUT}")


In [ ]:
# CELL 9: Cài đặt VieNeu-TTS + Evaluation dependencies
# VieNeu-TTS: clone repo + cài local
!git clone https://github.com/pnnbao97/VieNeu-TTS.git /content/VieNeu-TTS 2>/dev/null || (cd /content/VieNeu-TTS && git pull)
!pip install -q -e /content/VieNeu-TTS

# Eval dependencies
!pip install -q \
    soundfile \
    "ragas>=0.2.0" \
    langchain-google-genai \
    "faster-whisper>=1.0.0" \
    utmos


In [ ]:
# CELL 10: Khởi động TTS server (VieNeu-TTS)
import subprocess, sys, time, requests, os

PROJECT = "/content/legal-voice-callbot/nutrition-callbot"
os.system("pkill -f tts.server 2>/dev/null")
time.sleep(1)

# Thêm VieNeu-TTS vào PYTHONPATH để tts/core/synthesizer.py import được
tts_env = {
    **os.environ,
    "TTS_PORT": "50053",
    "PYTHONPATH": "/content/VieNeu-TTS:" + os.environ.get("PYTHONPATH", ""),
}

tts_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "tts.server:app",
     "--host", "0.0.0.0", "--port", "50053"],
    cwd=PROJECT,
    stdout=open("/content/tts.log", "w"),
    stderr=subprocess.STDOUT,
    env=tts_env,
)

# Chờ server sẵn sàng (model load lần đầu có thể mất 1-2 phút)
print("Đang chờ TTS server (lần đầu sẽ tải model ~1-2 phút)...")
for i in range(90):
    try:
        if requests.get("http://localhost:50053/health", timeout=2).status_code == 200:
            print(f"TTS server OK (PID={tts_proc.pid})")
            break
    except: pass
    time.sleep(3)
    if i == 89: print("TIMEOUT — xem tts.log")

!tail -15 /content/tts.log


In [ ]:
# CELL 11: Test end-to-end pipeline (Brain → TTS)
import requests, json, time, soundfile as sf, io
import IPython.display as ipd

QUERY = "Thưa bác sĩ, tôi muốn hỏi có nên uống Omega 3-6-9 mỗi ngày hay không?"

# 1. Lấy câu trả lời từ Brain (Gemini + RAG)
t_brain = time.perf_counter()
r = requests.post(
    "http://localhost:50052/think",
    json={"query": QUERY, "session_id": "e2e_test"},
)
brain_data = r.json()
answer = brain_data["text"]
brain_s = time.perf_counter() - t_brain
print(f"Brain: {brain_s:.2f}s")
print(f"Answer ({len(answer)} ký tự):\n{answer[:300]}...\n")

# 2. Tổng hợp giọng nói qua VieneuTTS
t_tts = time.perf_counter()
tts_r = requests.post(
    "http://localhost:50053/speak",
    json={"text": answer},
    timeout=120,
)
tts_s = time.perf_counter() - t_tts

ttfb_ms = float(tts_r.headers.get("X-TTFB-ms", 0))
rtf     = float(tts_r.headers.get("X-RTF", 0))
dur_s   = float(tts_r.headers.get("X-Duration-s", 0))

print(f"TTS: {tts_s:.2f}s | TTFB: {ttfb_ms:.0f}ms | RTF: {rtf:.3f} | Dài: {dur_s:.1f}s")
print(f"Tổng pipeline: {brain_s + tts_s:.2f}s")

# Phát audio trong Colab
audio, sr = sf.read(io.BytesIO(tts_r.content))
print(f"\nAudio ({sr}Hz, {len(audio)/sr:.1f}s):")
ipd.display(ipd.Audio(audio, rate=sr))


In [ ]:
# CELL 12: RAGAs Evaluation (Context Precision, Recall, Faithfulness,
#           Answer Relevancy, Factual Correctness)
import json, os

# Load eval results (từ Cell 8)
results = []
with open("/content/drive/MyDrive/Nutrition data/eval_results.jsonl") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

ok = [r for r in results if "error" not in r and r.get("contexts")]
print(f"Records hợp lệ (có contexts): {len(ok)}/{len(results)}")

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    ContextRecall,
    Faithfulness,
    ResponseRelevancy,
    FactualCorrectness,
)
from ragas.llms import LangchainLLMWrapper
from langchain_google_genai import ChatGoogleGenerativeAI

llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GEMINI_API_KEY"],
))

EVAL_N = 50  # dùng 50 mẫu để tiết kiệm quota
samples = [
    SingleTurnSample(
        user_input=r["question"],
        response=r["answer"],
        reference=r["ground_truth"],
        retrieved_contexts=r["contexts"],
    )
    for r in ok[:EVAL_N]
]
dataset = EvaluationDataset(samples=samples)

result = evaluate(
    dataset=dataset,
    metrics=[
        LLMContextPrecisionWithReference(),
        ContextRecall(),
        Faithfulness(),
        ResponseRelevancy(),
        FactualCorrectness(),
    ],
    llm=llm,
)
print(result)

OUT = "/content/drive/MyDrive/Nutrition data/ragas_results.csv"
result.to_pandas().to_csv(OUT, index=False)
print(f"Saved: {OUT}")


In [ ]:
# CELL 13: Latency Analysis — p50 / p90 / p95 / p99
import json, numpy as np

results = []
with open("/content/drive/MyDrive/Nutrition data/eval_results.jsonl") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

ok = [r for r in results if "error" not in r]

def pct_row(label, vals_ms):
    a = np.array(vals_ms)
    row = f"{label:<22s}"
    for p in [50, 90, 95, 99]:
        row += f"  p{p}={np.percentile(a,p):.0f}ms"
    row += f"  mean={a.mean():.0f}ms  n={len(a)}"
    print(row)

print("=" * 70)
print(f"{'Metric':<22s}  {'p50':>8s}  {'p90':>8s}  {'p95':>8s}  {'p99':>8s}  mean")
print("=" * 70)

latencies_ms = [r["latency_s"] * 1000 for r in ok]
pct_row("Total latency", latencies_ms)

ttfts_ms = [r["ttft_s"] * 1000 for r in ok if r.get("ttft_s")]
if ttfts_ms:
    pct_row("TTFT", ttfts_ms)

rag_ms = [r["timing"]["rag_ms"] for r in ok if r.get("timing", {}).get("rag_ms")]
if rag_ms:
    pct_row("RAG retrieval", rag_ms)

expand_ms = [r["timing"]["expand_ms"] for r in ok if r.get("timing", {}).get("expand_ms")]
if expand_ms:
    pct_row("Query expansion", expand_ms)

llm_ms = [r["timing"]["llm_ttft_ms"] for r in ok if r.get("timing", {}).get("llm_ttft_ms")]
if llm_ms:
    pct_row("LLM TTFT", llm_ms)

print("=" * 70)
errors = len(results) - len(ok)
print(f"Total records: {len(results)} | Errors: {errors} | Success: {len(ok)}")


In [ ]:
# CELL 14: TTS Quality Evaluation
#   - UTMOS: naturalness MOS proxy (không cần human judge)
#   - WER via faster-whisper: intelligibility
#   - TTFB + RTF: latency percentiles
import requests, numpy as np, soundfile as sf, io, json, time, torch

# Load 20 câu trả lời mẫu
results = []
with open("/content/drive/MyDrive/Nutrition data/eval_results.jsonl") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))
TEST_N = 20
test_samples = [r for r in results if "error" not in r and r.get("answer")][:TEST_N]

# ── 1. Tổng hợp audio + đo TTFB / RTF ──────────────────────────────────────
tts_stats = []
print(f"Synthesizing {len(test_samples)} samples...")
for i, rec in enumerate(test_samples):
    text = rec["answer"][:400]  # giới hạn độ dài
    r = requests.post("http://localhost:50053/speak", json={"text": text}, timeout=120)
    audio, sr = sf.read(io.BytesIO(r.content))
    tts_stats.append({
        "audio": audio, "sr": sr, "text": text,
        "ttfb_ms": float(r.headers.get("X-TTFB-ms", 0)),
        "rtf":     float(r.headers.get("X-RTF", 0)),
        "dur_s":   float(r.headers.get("X-Duration-s", 0)),
    })
    if (i+1) % 5 == 0: print(f"  {i+1}/{len(test_samples)}")

ttfbs = [s["ttfb_ms"] for s in tts_stats]
rtfs  = [s["rtf"]     for s in tts_stats]
print("\n=== TTS Latency ===")
for label, vals in [("TTFB (ms)", ttfbs), ("RTF", rtfs)]:
    a = np.array(vals)
    print(f"{label:<12s} p50={np.percentile(a,50):.1f}  p90={np.percentile(a,90):.1f}  p99={np.percentile(a,99):.1f}  mean={a.mean():.1f}")

# ── 2. UTMOS MOS (naturalness) ───────────────────────────────────────────────
try:
    import utmos
    predictor = utmos.score.load_model()
    mos_scores = []
    for s in tts_stats:
        wave_t = torch.FloatTensor(s["audio"]).unsqueeze(0)
        score  = predictor.score(wave_t, s["sr"])
        mos_scores.append(score.item())
    print(f"\nUTMOS MOS:  {np.mean(mos_scores):.3f} ± {np.std(mos_scores):.3f}")
    print(f"  min={min(mos_scores):.3f}  max={max(mos_scores):.3f}")
except Exception as e:
    print(f"\nUTMOS: {e}")

# ── 3. WER via faster-whisper (intelligibility) ──────────────────────────────
try:
    from faster_whisper import WhisperModel

    def word_error_rate(ref: str, hyp: str) -> float:
        import difflib
        r, h = ref.lower().split(), hyp.lower().split()
        ops   = difflib.SequenceMatcher(None, r, h).get_opcodes()
        edits = sum(max(o[2]-o[1], o[4]-o[3]) for o in ops if o[0] != "equal")
        return edits / max(1, len(r))

    whisper = WhisperModel("large-v3", device="cuda", compute_type="float16")
    wers = []
    for s in tts_stats[:10]:
        segs, _ = whisper.transcribe(s["audio"], language="vi")
        hyp = " ".join(seg.text for seg in segs)
        wers.append(word_error_rate(s["text"], hyp))
    print(f"WER (Intelligibility): {np.mean(wers):.3f} ± {np.std(wers):.3f}")
    print(f"  (WER thấp hơn = nghe rõ hơn)")
except Exception as e:
    print(f"WER: {e}")
